In [ ]:
import pandas as pd
from pathlib import Path
import re
from difflib import SequenceMatcher
from datetime import datetime
from unidecode import unidecode
from collections import Counter
from rapidfuzz import process, fuzz

cols_dig = ['applicantNumber', 'id_par', 'id_name_dig', 'requestDate', 'cod_vil', 'numParcelleOF_x', 'numParcelleOF_c', 'firstname', 'lastname', 'fullname',  'fullname_control', 'managerName', 'managerName_control', 'nameSignatory', 'nameSignatory_control', 'label','description_x','village','estimatedArea', 'identityDocumentNumber_x', 'phoneNumber_x','typeOfIndividualCertificate', 'nameOfGroup', 'nomPrenomCE', 'identityDocumentPhoto']
cols_ctb = ["id_par","id_name_ctb","nom_douar","code_parcelle", "cd_vil", "id_parcelle", "obs", "obs_control", "num_demande", "cin"]

def normalize_name(name):
    if pd.isna(name):
        return ""
    return '_'.join(str(name).lower().split()).strip()

def normalize(name):
    words = unidecode(str(name)).lower().split()
    words.sort()
    return "_".join(words)

def normalize_cols(df, cols):
    norm = (
        df[cols]
        .astype(str)
        .apply(lambda s: s.str.split('_').apply(lambda x: '_'.join(sorted(x))))
    )
    return norm.eq(norm.iloc[:, 0], axis=0).all(axis=1)

def best_match(x, choices):
    match, score, _ = process.extractOne(x, choices, scorer=fuzz.ratio)
    return pd.Series([match, score])

def similaire(a, b, seuil=0.8):
    a = normalize(a)
    b = normalize(b)
    ratio = SequenceMatcher(None, a, b).ratio()
    return ratio >= seuil

date = datetime.now()
date = date.strftime("%Y-%m-%d__%H-%M-%S")

# Initialisation of data
digifor_path = Path(r"C:\Users\L14\Downloads\ABH_CSV\DEMANDES_total_fusionne.xlsx")
ctb_path = Path(r"C:\Users\L14\Downloads\tonkpi_plle_info_12-03-2026_3.xlsx")
OUTPUT_FOLDER = Path(r"C:\Users\L14\Desktop\GESTION_DEMANDES")

df_digifor_owners = pd.read_excel(digifor_path, engine='openpyxl')
df_ctb_owners = pd.read_excel(ctb_path, engine='openpyxl')

df_digifor_owners.loc[:, "id_name_dig"] = ""
df_ctb_owners.loc[:, "id_name_ctb"] = ""
resum_data = []

# remove canceled 
pattern = r'annul|litige|erreur|anul|annl'
mask = (
    df_digifor_owners['fullname'].str.contains(pattern, case=False, na=False)
    |
    df_digifor_owners['numParcelleOF_c'].str.contains(pattern, case=False, na=False)
)


df_digifor_owners_canceled = df_digifor_owners[mask]
df_digifor_owners_canceled.to_excel(f"{OUTPUT_FOLDER}/1__demandes_annulees_{date}.xlsx")
nb_digifor_owners_canceled = df_digifor_owners_canceled.shape[0]

df_digifor_owners = df_digifor_owners[~mask].copy()

#applicantNumber	requestDate	numParcelleOF_x	numParcelleOF_c	firstname	lastname	fullname	label	description	village	estimatedArea	identityDocumentNumber	phoneNumber	typeOfIndividualCertificate	nameOfGroup	nomPrenomCE	identityDocumentPhoto

# Formating data for comparaison
df_digifor_owners.loc[:, "cod_vil"] = (
    df_digifor_owners["cod_vil"].astype(str)
    .apply(lambda x : unidecode(x) if pd.notna(x) else x)
    .str.lower()
    .str.strip()
)

df_ctb_owners.loc[:, "cd_vil"] = (
    df_ctb_owners["cd_vil"].astype(str)
    .apply(lambda x : unidecode(x) if pd.notna(x) else x)
    .str.lower()
    .str.strip()
)

df_digifor_owners.loc[:, "fullname_control"] = (
    df_digifor_owners["fullname"].astype(str)
    .apply(lambda x : normalize_name(x) if pd.notna(x) else x)
)

df_digifor_owners.loc[:, "managerName_control"] = (
    df_digifor_owners["managerName"].astype(str)
    .apply(lambda x : normalize_name(x) if pd.notna(x) else x)
)

df_digifor_owners.loc[:, "nameSignatory_control"] = (
    df_digifor_owners["nameSignatory"].astype(str)
    .apply(lambda x : normalize_name(x) if pd.notna(x) else x)
)

df_ctb_owners.loc[:, "obs_control"] = (
    df_ctb_owners["obs"].astype(str)
    .replace(r"\((.*)\)", "", regex=True)
    .replace(r"Heritiers de ", "", regex=True)
    .apply(lambda x : normalize_name(x) if pd.notna(x) else x)
)

# Create merging keys

df_digifor_owners.loc[:, "id_par"] = (
    df_digifor_owners["cod_vil"] + "__" + df_digifor_owners["numParcelleOF_c"]
)

df_ctb_owners.loc[:, "id_par"] = (
    df_ctb_owners["cd_vil"] + "__" + df_ctb_owners["code_parcelle"]
)

df_ctb_owners.loc[:, "id_par_obs"] = (
    df_ctb_owners["id_parcelle"].astype(str) + "__" + df_ctb_owners["obs"].astype(str)
)
# Retrieve and drop duplicated data
df_ctb_owners = df_ctb_owners.drop_duplicates(subset='id_par_obs', keep='first')

df_digifor_owners_duplicated = df_digifor_owners[df_digifor_owners.duplicated(subset='id_par', keep=False)]
df_ctb_owners_duplicated = df_ctb_owners[df_ctb_owners.duplicated(subset='id_par', keep=False)]
df_digifor_owners_duplicated_dem = df_digifor_owners[df_digifor_owners.duplicated(subset='applicantNumber', keep=False)]


df_digifor_owners_duplicated_dem.to_excel(f"{OUTPUT_FOLDER}/2__demandes_dupliquees_{date}.xlsx")
df_ctb_owners_duplicated.to_excel(f"{OUTPUT_FOLDER}/3__parcelles_dupliquees_{date}.xlsx")
df_digifor_owners_duplicated.to_excel(f"{OUTPUT_FOLDER}/4__demandes_avec_erreur_affectation_parcelle_{date}.xlsx")

df_digifor_owners = df_digifor_owners.drop_duplicates(subset='id_par')
df_ctb_owners = df_ctb_owners.drop_duplicates(subset='id_par')

#nom_douar	cd_vil	id_parcelle	code_parcelle	obs	cin	num_demande	ouv_pub	observation	old_id_parcelle	surface_cal

# 1- Detecting land on the same village and compare the name 
df_owners = df_digifor_owners.merge(
    df_ctb_owners[cols_ctb],
    on='id_par',
    how='outer',
    indicator=True
) 

df_owners_inner = df_owners[df_owners["_merge"] == "both"].copy()
df_digifor_owners = df_owners[df_owners["_merge"] == "left_only"].copy()
df_ctb_owners = df_owners[df_owners["_merge"] == "right_only"].copy()

df_digifor_owners = df_digifor_owners.loc[:, cols_dig]
df_ctb_owners = df_ctb_owners.loc[:, cols_ctb]

mask_gestionnaire = df_owners_inner['managerName'].astype(str) == ""
mask_detenteur_droit = df_owners_inner['nameSignatory'].astype(str) == ""

#df_gestionnaire = df_owners_inner.loc[mask_gestionnaire].copy()
#df_detenteur = df_owners_inner.loc[~mask_gestionnaire & mask_detenteur_droit].copy()
#df_normal = df_owners_inner.loc[~mask_gestionnaire & ~mask_detenteur_droit].copy()

df_owners_inner.loc[:, 'same_name_dem'] = (
    df_owners_inner.loc[:,'fullname_control'].astype(str)
    .str.split('_')
    .apply(sorted)
    ==
    df_owners_inner.loc[:,'obs_control'].astype(str)
    .str.split('_')
    .apply(sorted)
)

df_owners_inner.loc[:, 'same_name_ges'] = (
    df_owners_inner.loc[:,'managerName_control'].astype(str)
    .str.split('_')
    .apply(sorted)
    ==
    df_owners_inner.loc[:,'obs_control'].astype(str)
    .str.split('_')
    .apply(sorted)
)

df_owners_inner.loc[:, 'same_name_deten'] = (
    df_owners_inner.loc[:,'nameSignatory_control'].astype(str)
    .str.split('_')
    .apply(sorted)
    ==
    df_owners_inner.loc[:,'obs_control'].astype(str)
    .str.split('_')
    .apply(sorted)
)

#df_gestionnaire["same_name"] = normalize_cols(df_gestionnaire,["fullname_control","managerName_control","nameSignatory_control","obs_control"])

#df_detenteur["same_name"] = normalize_cols(df_detenteur,["fullname_control","nameSignatory_control","obs_control"])

#df_normal["same_name"] = normalize_cols(df_normal,["fullname_control","obs_control"])

#df_owners_inner = pd.concat([df_gestionnaire, df_detenteur, df_normal])

cols = ["same_name_dem", "same_name_ges", "same_name_deten"]
cols_sim = ["same_name_dem_sim", "same_name_ges_sim", "same_name_deten_sim"]

df_owners_inner_same_name = df_owners_inner[df_owners_inner[cols].any(axis=1)]
df_owners_inner_same_name.to_excel(f"{OUTPUT_FOLDER}/5__parcelles_demandes_avec_noms_corrects_{date}.xlsx")

df_owners_inner_diff_name = df_owners_inner[~df_owners_inner[cols].any(axis=1)]
df_digifor_owners_rest = df_owners_inner_diff_name.loc[:,cols_dig]

df_ctb_owners_rest = df_owners_inner_diff_name.loc[:,cols_ctb]

df_digifor_owners = pd.concat([df_digifor_owners, df_digifor_owners_rest])
df_ctb_owners = pd.concat([df_ctb_owners, df_ctb_owners_rest])

# 2 - Detecting land on the same village and compare the name
df_digifor_owners["id_name_dig"] = (
    df_digifor_owners["cod_vil"] + "__" + df_digifor_owners["fullname_control"]
)

df_digifor_owners.loc[:, "id_name_deten"] = (
    df_digifor_owners["cod_vil"] + "__" + df_digifor_owners["nameSignatory_control"])

df_ctb_owners["id_name_ctb"] = (
    df_ctb_owners["cd_vil"] + "__" + df_ctb_owners["obs_control"]
)
 
df_owners_name = df_digifor_owners.merge(
    df_ctb_owners[cols_ctb],
    left_on='id_name_dig',
    right_on='id_name_ctb',
    how='outer',
    indicator= "_second_merge"
) 

df_owners_name_inner = df_owners_name[df_owners_name["_second_merge"] == "both"].copy()

df_digifor_owners = df_owners_name[df_owners_name["_second_merge"] == "left_only"].copy()
df_ctb_owners = df_owners_name[df_owners_name["_second_merge"] == "right_only"].copy()

df_digifor_owners.rename(columns={'id_par_x':'id_par'},inplace=True)
df_ctb_owners.rename(columns={'id_par_x':'id_par'},inplace=True)

df_digifor_owners = df_digifor_owners.loc[:, cols_dig]
df_ctb_owners = df_ctb_owners.loc[:, cols_ctb]

df_owners_name = df_digifor_owners.merge(
    df_ctb_owners[cols_ctb],
    left_on='id_name_dig',
    right_on='id_name_ctb',
    how='outer',
    indicator= "_third_merge"
) 

df_owners_name_inner_deten = df_owners_name[df_owners_name["_third_merge"] == "both"].copy()

df_owners_name_inner.to_excel(f"{OUTPUT_FOLDER}/6__parcelles_demandes_differents_avec_memes_noms_{date}.xlsx")
df_owners_name_inner_deten.to_excel(f"{OUTPUT_FOLDER}/7__parcelles_demandes_differents_avec_memes_noms_detenteurs_{date}.xlsx")


#df_owners_inner_diff_name.to_excel(f"{OUTPUT_FOLDER}/parcelles_demandes_avec_noms_incorrects_{date}.xlsx")
df_digifor_owners = df_owners_name[df_owners_name["_third_merge"] == "left_only"].copy()

df_ctb_owners = df_owners_name[df_owners_name["_third_merge"] == "right_only"].copy()

df_digifor_owners.rename(columns={'id_par_x':'id_par'},inplace=True)
df_ctb_owners.rename(columns={'id_par_x':'id_par'},inplace=True)
df_digifor_owners = df_digifor_owners.loc[:, cols_dig]
df_ctb_owners = df_ctb_owners.loc[:, cols_ctb]

df_owners = df_digifor_owners.merge(
    df_ctb_owners[cols_ctb],
    on='id_par',
    how='outer',
    indicator= "_four_merge"
) 

df_owners_inner = df_owners[df_owners["_four_merge"] == "both"].copy()
df_digifor_owners = df_owners[df_owners["_four_merge"] == "left_only"].copy()
df_ctb_owners = df_owners[df_owners["_four_merge"] == "right_only"].copy()

df_digifor_owners = df_digifor_owners.loc[:, cols_dig]
df_ctb_owners = df_ctb_owners.loc[:, cols_ctb]

df_owners_inner["same_name_dem_sim"] = [
    similaire(a, b)
    for a, b in zip(
        df_owners_inner["fullname_control"],
        df_owners_inner["obs_control"]
    )
]

df_owners_inner["same_name_deten_sim"] = [
    similaire(a, b)
    for a, b in zip(
        df_owners_inner["nameSignatory_control"],
        df_owners_inner["obs_control"]
    )
]

df_owners_inner["same_name_ges_sim"] = [
    similaire(a, b)
    for a, b in zip(
        df_owners_inner["managerName_control"],
        df_owners_inner["obs_control"]
    )
]

df_owners_inner_same_name_sim = df_owners_inner[df_owners_inner[cols_sim].any(axis=1)]
df_owners_inner_same_name_sim.to_excel(f"{OUTPUT_FOLDER}/8__parcelles_demandes_avec_noms_corrects_avec_similutes_sup_80_{date}.xlsx")

df_owners_inner_diff_name_sim = df_owners_inner[
    ~df_owners_inner[cols_sim].any(axis=1)
]
df_digifor_owners_rest = df_owners_inner_diff_name_sim[cols_dig]

df_ctb_owners_rest = df_owners_inner_diff_name_sim[cols_ctb]

#df_owners_inner_diff_name_sim.to_excel(f"7bar__{OUTPUT_FOLDER}/parcelles_demandes_avec_noms_incorrects_avec_similutes_sup_80_{date}.xlsx")

df_digifor_owners = pd.concat([df_digifor_owners,df_digifor_owners_rest ])
df_ctb_owners = pd.concat([df_ctb_owners, df_ctb_owners_rest])

df_digifor_owners_grouped = {
    k: v for k, v in df_digifor_owners.groupby("cod_vil")
}
df_ctb_owners_grouped = {
    k: v for k, v in df_ctb_owners.groupby("cd_vil")
}

results = []

villages = set(df_digifor_owners_grouped.keys()) & set(df_ctb_owners_grouped.keys())
matched_ctb_indices = set()
matched_dig_indices = set()

for vil in villages:

    df_dig = df_digifor_owners_grouped[vil]
    df_ctb = df_ctb_owners_grouped[vil]

    used_indices = set()
    
    for _, row_big in df_dig.iterrows():

        available_small = df_ctb[~df_ctb.index.isin(used_indices)]
        
        
        if available_small.empty:
            break

        choices = available_small["obs_control"].tolist()

        match = process.extractOne(
            row_big["fullname_control"],
            choices,
            scorer=fuzz.token_sort_ratio
        )

        if match is None:
            continue

        match_name, score, pos = match

        row_small = available_small.iloc[pos]
        idx_small = available_small.index[pos]

        used_indices.add(idx_small)
        matched_ctb_indices.add(idx_small)
        matched_dig_indices.add(row_big.name)

        combined = {
            "id_parcelle": row_small["id_parcelle"],
            "applicantNumber": row_big["applicantNumber"],
            "managerName": row_big["managerName"],
            "nameSignatory": row_big["nameSignatory"],
            "requestDate": row_big["requestDate"],
            "village": vil,
            "digifor_name": row_big["fullname"],
            "ctb_name": row_small["obs"],
            "num_demande": row_small["num_demande"],
            "digifor_parcelle": row_big["numParcelleOF_c"],
            "ctb_parcelle": row_small["code_parcelle"],
            "similarity": score
        }

        results.append(combined)

df_result = pd.DataFrame(results)
reste_ctb = df_ctb_owners.loc[~df_ctb_owners.index.isin(matched_ctb_indices)]
reste_dig = df_digifor_owners.loc[~df_digifor_owners.index.isin(matched_dig_indices)]

df_result.to_excel(f"{OUTPUT_FOLDER}/9__matching_digifor_ctb_{date}.xlsx")

reste_ctb.to_excel(f"{OUTPUT_FOLDER}/10__ctb_sans_correspondance_{date}.xlsx")

reste_dig.to_excel(f"{OUTPUT_FOLDER}/11__digifor_sans_correspondance_{date}.xlsx")


Index(['Unnamed: 0', 'applicantNumber', 'requestDate', 'numParcelleOF_x',
       'numParcelleOF_c', 'firstname', 'lastname', 'fullname', 'label',
       'description', 'village', 'estimatedArea', 'identityDocumentNumber',
       'phoneNumber', 'typeOfIndividualCertificate', 'nameOfGroup',
       'nomPrenomCE', 'identityDocumentPhoto'],
      dtype='object')
Index(['nom_douar', 'cd_vil', 'id_parcelle', 'code_parcelle', 'obs', 'cin',
       'num_demande', 'ouv_pub', 'observation', 'old_id_parcelle',
       'surface_cal'],
      dtype='object')